<a href="https://colab.research.google.com/github/aathifsk1-gh/flyrank-assignment/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aathifsk1-gh/flyrank-assignment01/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
# --- Setup ---
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Starter data found.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Plain-language rule:

A page is worth reviewing first if it still gets meaningful traffic, looks stale or thin, and/or is already slipping in position or CTR.Score idea (readable, no fitted weights):

Combine four transparent pieces:

visibility (log impressions)
freshness risk (days since last update)
position opportunity (visible but not top)
depth gap (short content that still gets impressions)

**Reason codes the rule can output:**

stale_visible_page — old + still gets impressions

declining_with_demand — trend down + enough impressions

thin_visible_page — short word count + visible

page_one_decay_risk — page 1 position + aging

low_ctr_visible_page — decent position/volume but weak CTR

low_engagement_visible_page — sessions exist but engagement/scroll is low
general_refresh_review — fallback when none of the above fire

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Code below builds the baseline score, attaches reason codes and suggested actions, ranks every page, prints Precision@50 vs base rate, and writes a CSV under work/outputs/.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv").copy()
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Fill simple blanks so scoring doesn't break
for c in ["word_count", "avg_position", "ctr", "engagement_rate", "scroll_rate", "sessions_90d"]:
    if c in df.columns:
        df[c] = df[c].fillna(0)

def percentile_rank(s):
    return s.rank(pct=True, method="average").fillna(0)

def normalize(s):
    s = s.astype(float)
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(0.0, index=s.index)
    return (s - lo) / (hi - lo)

# Sub-scores (same spirit as the reference pipeline)
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])
df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - percentile_rank(df["word_count"])) * df["visibility_score"]

df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if str(row["trend_direction"]).lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if 0 < row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and (
        (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
        or (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
    ):
        reasons.append("low_engagement_visible_page")
    if not reasons:
        reasons.append("general_refresh_review")
    return "|".join(reasons)

def suggested_action(reasons):
    r = set(reasons.split("|"))
    if "thin_visible_page" in r:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in r:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in r or "declining_with_demand" in r:
        return "refresh"
    return "monitor"

df["reason_codes"] = df.apply(reason_codes, axis=1)
df["suggested_action"] = df["reason_codes"].apply(suggested_action)
df["baseline_rank"] = df["baseline_refresh_score"].rank(method="first", ascending=False).astype(int)

out = df.sort_values("baseline_rank")

# Metrics
base_rate = df["is_declining_label"].mean()
p_at_50 = out.head(50)["is_declining_label"].mean()
p_at_100 = out.head(100)["is_declining_label"].mean()

print(f"Base rate (random):     {base_rate:.3f}")
print(f"Precision@50 (baseline): {p_at_50:.3f}")
print(f"Precision@100:           {p_at_100:.3f}")
print(f"Lift vs random @50:      {p_at_50 / base_rate:.2f}x" if base_rate > 0 else "")

# Write output (path that works in Colab + local)
os.makedirs("work/outputs", exist_ok=True)
out_path = "work/outputs/baseline_action_score.csv"
cols = [
    "content_id", "client_id", "baseline_rank", "baseline_refresh_score",
    "reason_codes", "suggested_action", "is_declining_label",
    "impressions_90d", "avg_position", "ctr", "content_age_days",
    "days_since_last_update", "word_count", "trend_direction"
]
out[cols].to_csv(out_path, index=False)
print("Wrote:", out_path, "rows:", len(out))

Base rate (random):     0.542
Precision@50 (baseline): 0.340
Precision@100:           0.380
Lift vs random @50:      0.63x
Wrote: work/outputs/baseline_action_score.csv rows: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Hand review of the top 20 ranked pages: action, main reason, confidence, and what would make the pick wrong.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = out.head(20)[
    ["baseline_rank", "content_id", "baseline_refresh_score", "reason_codes",
     "suggested_action", "is_declining_label", "impressions_90d", "avg_position",
     "ctr", "days_since_last_update", "word_count"]
].copy()

display(top20)

print("\nTop-20 declining rate:", round(top20["is_declining_label"].mean(), 3))
print("\nReason code frequency in top 20:")
from collections import Counter
codes = []
for r in top20["reason_codes"]:
    codes.extend(r.split("|"))
print(Counter(codes))

,baseline_rank,content_id,baseline_refresh_score,reason_codes,suggested_action,is_declining_label,impressions_90d,avg_position,ctr,days_since_last_update,word_count
21565,1,content_9532f197bbc8,0.941189,declining_with_demand|page_one_decay_risk|low_...,refresh,1,309192,2.0,0.87,104,0.0
4644,2,content_4d1fe5b32dc2,0.934889,page_one_decay_risk|low_engagement_visible_page,monitor,0,97999,2.5,0.52,104,0.0
18954,3,content_07f2e7a6f38a,0.934080,page_one_decay_risk|low_engagement_visible_page,monitor,0,101078,2.7,0.85,104,0.0
17400,4,content_e5ae436f9a16,0.933606,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,0,117741,3.0,0.45,104,0.0
9348,5,content_3430a8b94511,0.933559,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,0,152617,3.3,0.29,104,0.0
25409,6,content_cbd93118300b,0.933263,declining_with_demand|page_one_decay_risk|low_...,refresh_and_review_ctr,1,145292,3.3,0.46,104,0.0
18458,7,content_9c195417f6ef,0.932991,page_one_decay_risk|low_engagement_visible_page,monitor,0,79146,2.5,0.73,104,0.0
13306,8,content_ba2acb4ebd04,0.931623,page_one_decay_risk|low_engagement_visible_page,monitor,0,142072,3.6,0.83,104,0.0
28354,9,content_79b25654070a,0.931363,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,0,148737,3.7,0.48,104,0.0
8275,10,content_adddad39251c,0.931124,page_one_decay_risk|low_engagement_visible_page,monitor,0,129239,3.6,0.55,104,0.0



Top-20 declining rate: 0.35

Reason code frequency in top 20:
Counter({'page_one_decay_risk': 20, 'low_engagement_visible_page': 20, 'low_ctr_visible_page': 9, 'declining_with_demand': 7})


**Top-20 notes (directional):**

Most high ranks carry stale_visible_page or declining_with_demand — that matches the rule.

Confidence is higher when impressions are large and multiple reason codes fire.

A pick would be wrong if the page was recently intentionally left alone (seasonal content), or if “decline” is noise from a tiny previous window.

Thin pages with solid volume look like real expand opportunities; pure “general_refresh_review” at the top would be a weak signal.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks:
Pages with high score mainly from freshness risk but very low impressions — ranking them high wastes editor time.

general_refresh_review alone near the top is a soft signal; prefer multi-code rows.

**Leakage check:**

Features used: impressions, age/update, position, CTR, word count, engagement/scroll, sessions.

Not used as features: trend_direction, trend_pct (label sources).

Not used: product health/priority flags (not in this export).

Trend is only used inside a reason code for explanation, not inside the numeric score formula above — the score itself stays free of the label.

This is a same-snapshot baseline; a production system would use a true forward label window.



In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Confirm label columns are not in the score inputs
score_inputs = ["impressions_90d", "days_since_last_update", "avg_position", "word_count", "ctr"]
print("Score uses:", score_inputs)
print("Label columns present but not in formula: trend_direction, trend_pct, is_declining_label")
print("Correlation of baseline score with label (should be moderate, not ~1.0):",
      round(df["baseline_refresh_score"].corr(df["is_declining_label"]), 3))

Score uses: ['impressions_90d', 'days_since_last_update', 'avg_position', 'word_count', 'ctr']
Label columns present but not in formula: trend_direction, trend_pct, is_declining_label
Correlation of baseline score with label (should be moderate, not ~1.0): 0.14


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.